# Visualization of Cell2location deconvolution os ASAP samples
* This notebook helps to visualize each ASAP Visium sample deconvoluted using [Cell2Location](https://cell2location.readthedocs.io/en/latest/) by the [spatial-sc-rnaseq-integration-wf](https://github.com/ASAP-CRN/spatial-sc-rnaseq-integration-wf) workflow.
* If local files exist in the `_LOCAL_CACHE_DIR` directory, they will be loaded. Otherwise they will be downloaded from the `_DATASET_GCS_DIR` (`gcloud` and bucket read credentials are needed).
* Cell metadata (i.e. `obs`) include `cell2location_leiden_res_*` columns indicating Leiden clusters from the cell type deconvolution vectors--not from gene expression.
  
#### Notes:
  * Some raw bucket sample_id's have a typo PM**BD**S instead of PM**DB**S. This is handled by `_raw_sample_name_candidates()`.
  * The ASAP `scherzer-pmdbs-spatial-visium-mtg` dataset spatial samples are NOT expected to have dopaminergic (DA) neurons because they are from Middle temporal gyrus (MTG, UBERON:0002771)--not from substantia nigra, which is where DA neurons are typically found. Deconvolution results should always be veryfied by cell type marker gene expression.

## Imports

In [ ]:
import glob
import re
import subprocess
import sys
import warnings
from pathlib import Path

from IPython.display import Image, HTML, display
import ipywidgets as widgets

import anndata as ad
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns
import squidpy as sq
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

pio.renderers.default = "plotly_mimetype"
sc.set_figure_params(dpi=100, frameon=False)
warnings.filterwarnings("ignore", message=".*deprecated.*")

## Inputs

In [ ]:
repo_name = "spatial-sc-rnaseq-integration-wf"  # needed for get_repo_root to work in both unpromoted and live repo notebooks
sys.path.insert(0, str(Path.cwd().parent / "utils"))
from notebook_utils import ensure_cached, get_repo_root, list_available_and_cached # must come after the utils sys.path.insert above
repo_root = get_repo_root(repo_name)
from input_definitions import (
    ASAP_RAW_H5AD_SUFFIX,
    DECONVOLUTED_H5AD_SUFFIX,
    asap_raw_gcs_dir,
    asap_raw_local_cache_dir,
    asap_release_gcs_dir,
    asap_strategy3_local_cache_dir,
)

# Google Cloud bucket -- note: needs `gcloud` installed and credentials to access bucket. Set to `uat` (will be changed to `curated` upon release)
_BUCKET_ENV = "uat"
_CRN_RELEASE_VERSION = "v6.0.0"
_DATASET_GCS_DIR = asap_release_gcs_dir(_BUCKET_ENV, _CRN_RELEASE_VERSION)

# Raw/all-genes H5ADs--not only HVG filtered.
_RAW_DATASET_GCS_DIR = asap_raw_gcs_dir()
_RAW_LOCAL_CACHE_DIR = asap_raw_local_cache_dir(repo_root)
RAW_H5AD_SUFFIX = ASAP_RAW_H5AD_SUFFIX

# The raw bucket's preprocess/counts_to_adata files use the ORIGINAL sample
# name (including "PMBDS" typo) and it's corrected in this notebook.
_PMDBS_PATTERN = re.compile(r"^ASAP_PMDBS_")

def _raw_sample_name_candidates(sample_name):
    """
    Return candidate raw-bucket sample names for `sample_name`, most likely first.

    Parameters
    ----------
    sample_name : str
        Sample ID in its promoted/output (PMDBS) spelling.

    Returns
    -------
    list of str
        [likely-original-PMBDS-spelling, sample_name] if they differ, else just `[sample_name]`.
    """
    candidates = [_PMDBS_PATTERN.sub("ASAP_PMBDS_", sample_name)]
    if sample_name not in candidates:
        candidates.append(sample_name)
    return candidates

# A prompt will be used to let user decide if these many samples should be downloaded
# from the Google bucket if not in cache.
N_SAMPLES_TO_DOWNLOAD = 3

# Pre-existing local cache
_LOCAL_CACHE_DIR = asap_strategy3_local_cache_dir(repo_root)

H5AD_SUFFIX = DECONVOLUTED_H5AD_SUFFIX
BASE_INDIR = _LOCAL_CACHE_DIR

# Prompt user to decide if cached (if any) should be used or download more
available_samples, cached_samples, missing_samples = list_available_and_cached(_DATASET_GCS_DIR, BASE_INDIR, H5AD_SUFFIX)
print(f"Found {len(available_samples)} sample(s) in {_DATASET_GCS_DIR}")
print(f"{len(cached_samples)}/{len(available_samples)} sample(s) already cached locally in {BASE_INDIR}")

_download_output = widgets.Output()

def _download_missing(_=None):
    _download_output.clear_output(wait=True)
    with _download_output:
        to_download = missing_samples[:N_SAMPLES_TO_DOWNLOAD]
        for s in to_download:
            ensure_cached(_DATASET_GCS_DIR, BASE_INDIR, s + H5AD_SUFFIX)
        print(f"Downloaded {len(to_download)}/{len(missing_samples)} not-yet-cached sample(s)")
        if len(missing_samples) > len(to_download):
            print(f"{len(missing_samples) - len(to_download)} sample(s) remain not cached "
                  "(increase N_SAMPLES_TO_DOWNLOAD and re-run this cell to fetch more)")

def _skip_download(_=None):
    _download_output.clear_output(wait=True)
    with _download_output:
        print(f"Skipped -- using the {len(cached_samples)} already-cached sample(s) only.")

if missing_samples:
    _n_offered = min(N_SAMPLES_TO_DOWNLOAD, len(missing_samples))
    _yes_button = widgets.Button(description="Yes", button_style="success")
    _no_button = widgets.Button(description="No", button_style="")
    _yes_button.on_click(_download_missing)
    _no_button.on_click(_skip_download)
    display(widgets.HBox([
        widgets.Label(f"Download {_n_offered} of {len(missing_samples)} missing sample(s) now?"),
        _yes_button, _no_button,
    ]))
    display(_download_output)
else:
    print("All samples already cached locally.")

## Utils

In [ ]:
sys.path.insert(0, str(repo_root / "utils"))
from genomics_constants import KAMATH_DA_MARKER_SYMBOLS
from notebook_utils import (
    coarse_category as uber_type_of,
    dropdown,
    ensure_uber_colors,
    golden_angle_palette,
    inject_big_dropdown_css,
    load_kamath_collapse_and_palette,
    selected_cell_types,
    spatial_scatter_grid,
    strip_q05_abundance_prefix,
    spatial_plot_genes,
)

inject_big_dropdown_css()

DA_UBER_CATEGORIES = {"CALB1", "SOX6"}

# Re-check the local cache here
cached_samples = sorted(s for s in available_samples if Path(BASE_INDIR, s + H5AD_SUFFIX).exists())
sample_dropdown = dropdown(big=True, options=cached_samples, description="Sample:")

# Used by selected_cell_types(), which Section 2 calls to restrict to CALB1_*/SOX6_* cell types.
cell_type_view_dropdown = dropdown(big=True, options=["DAs only","All cell types"], description="Show:")

# Used by Section 2's spatial_scatter_grid() call to toggle per-panel vs. shared vmax.
vmax_scale_dropdown = dropdown(
    big=True,
    options=["Shared across cell types", "Each cell type"],
    description="Scale ceiling:",
)

# Kamath cell-type-to-uber-category collapse dict/palette, loaded once.
_, uber_categories, uber_palette = load_kamath_collapse_and_palette(repo_root)

_MARKERS_TO_PLOT = KAMATH_DA_MARKER_SYMBOLS

## 1 — Exploration of each sample (select a sample)
In the dropdowns that will appear below, select: i) a sample, ii) either `All cell types` or `DAs only`, and iii) Section 2's colour-scale ceiling (`Each cell type` vs `Shared across cell types`)

In [ ]:
display(sample_dropdown)
display(cell_type_view_dropdown)
display(vmax_scale_dropdown)

In [ ]:
SAMPLE_NAME = sample_dropdown.value
H5AD_PATH = ensure_cached(_DATASET_GCS_DIR, BASE_INDIR, SAMPLE_NAME + H5AD_SUFFIX)

print(f"Loading: {H5AD_PATH}")
adata = ad.read_h5ad(H5AD_PATH)
print(adata.shape)

cell_types = list(adata.obsm["q05_cell_abundance_w_sf"].columns)
leiden_keys = sorted(c for c in adata.obs.columns if "leiden_res_" in c)  # cell2location_leiden_res_* for Strategy 3
frac_cols = [f"frac_{ct}" for ct in cell_types]
cell_types_ = [strip_q05_abundance_prefix(ct) for ct in cell_types]
ensure_uber_colors(cell_types_, uber_palette)

## 2 — Spatial cell type abundances
q05 posterior abundance per spot, one panel per cell type.

In [ ]:
for ct, ct_ in zip(cell_types, cell_types_):
    adata.obs[ct_] = adata.obsm["q05_cell_abundance_w_sf"][ct].values

cell_types_sel, cell_types_sel_ = selected_cell_types(cell_types, cell_types_, cell_type_view_dropdown.value, DA_UBER_CATEGORIES)

# None renders a plain tissue-only panel first (no color overlay); needs its
# own explicit title since there's no column/gene name to fall back to.
_panel_columns = [None] + cell_types_sel_
_panel_titles = ["Tissue only"] + cell_types_sel_

fig = spatial_scatter_grid(
    adata, _panel_columns, titles=_panel_titles,
    title_color_fn=lambda col: uber_palette[uber_type_of(col)],
    shared_vmax=(vmax_scale_dropdown.value == "Shared across cell types"),
)
if fig is not None:
    display(fig)
    plt.close(fig)

## 3 — Deconvolution Leiden clusters

In [ ]:
sc.pl.spatial(
    adata,
    color=leiden_keys,
    alpha=0.6,
    ncols=4,
    size=1.3,
    img_key="hires",
)

leiden_key = leiden_keys[-1]  # highest resolution available; change the index to inspect others

mean_frac = (
    adata.obs[[leiden_key] + frac_cols]
    .groupby(leiden_key)[frac_cols]
    .mean()
    .rename(columns=strip_q05_abundance_prefix)
)

mean_frac.columns = pd.MultiIndex.from_tuples(
    [(uber_type_of(ct), ct) for ct in mean_frac.columns],
    names=["uber_type", "cell_type"],
)
col_colors = [uber_palette[u] for u in mean_frac.columns.get_level_values("uber_type")]

fig, ax = plt.subplots(figsize=(12, max(4, len(mean_frac) * 0.5)))
mean_frac.plot(kind="barh", stacked=True, ax=ax, color=col_colors, legend=False)
ax.set_xlabel("Mean fraction")
ax.set_ylabel(leiden_key)
ax.set_title(f"Cell type composition per Leiden cluster ({leiden_key})")

legend_handles = [mpatches.Patch(color=uber_palette[u], label=u) for u in uber_categories]
ax.legend(handles=legend_handles, bbox_to_anchor=(1.01, 1), loc="upper left", fontsize=8, title="Cell type")

plt.tight_layout()
plt.show()

## 4 — Cell type markers

In [ ]:
# Need to load the H5AD's with all genes--not only HVG filtered
def _resolve_cached_raw_path(sample_name):
    """
    Return the local path for `sample_name`'s raw H5AD if already cached
    under either PMBDS/PMDBS spelling, else None.

    Parameters
    ----------
    sample_name : str
        Sample ID in its promoted/output (PMDBS) spelling.

    Returns
    -------
    pathlib.Path or None
        Local path if already cached, else None.
    """
    for candidate in _raw_sample_name_candidates(sample_name):
        local_path = Path(_RAW_LOCAL_CACHE_DIR, candidate + RAW_H5AD_SUFFIX)
        if local_path.exists():
            return local_path
    return None


def _download_raw_h5ad(sample_name):
    """
    Download `sample_name`'s raw H5AD, trying both PMBDS/PMDBS spellings.

    Parameters
    ----------
    sample_name : str
        Sample ID in its promoted/output (PMDBS) spelling.

    Returns
    -------
    pathlib.Path
        Local path to the now-cached file.

    Raises
    ------
    FileNotFoundError
        Neither spelling exists in the raw bucket.
    """
    for candidate in _raw_sample_name_candidates(sample_name):
        try:
            return ensure_cached(_RAW_DATASET_GCS_DIR, _RAW_LOCAL_CACHE_DIR, candidate + RAW_H5AD_SUFFIX)
        except subprocess.CalledProcessError:
            continue
    raise FileNotFoundError(f"No raw H5AD found for {sample_name} under either PMBDS/PMDBS spelling.")


def _plot_markers(raw_spatial_path):
    """
    Load `raw_spatial_path` and plot marker gene expression (raw UMI + log1p-normalized).

    Parameters
    ----------
    raw_spatial_path : pathlib.Path
        Local path to the raw (all-genes) H5AD.

    Returns
    -------
    None
    """
    print(f"Loading: {raw_spatial_path}")
    raw_adata = ad.read_h5ad(raw_spatial_path)
    spatial_plot_genes(raw_adata, _MARKERS_TO_PLOT, SAMPLE_NAME)


_raw_cached_path = _resolve_cached_raw_path(SAMPLE_NAME)

if _raw_cached_path is not None:
    _plot_markers(_raw_cached_path)
else:
    _raw_download_output = widgets.Output()

    def _download_and_plot_markers(_=None):
        _raw_download_output.clear_output(wait=True)
        with _raw_download_output:
            _plot_markers(_download_raw_h5ad(SAMPLE_NAME))

    def _skip_raw_download(_=None):
        _raw_download_output.clear_output(wait=True)
        with _raw_download_output:
            print(f"Skipped -- {SAMPLE_NAME}'s raw H5AD was not downloaded, markers not plotted.")

    _raw_yes_button = widgets.Button(description="Yes", button_style="success")
    _raw_no_button = widgets.Button(description="No", button_style="")
    _raw_yes_button.on_click(_download_and_plot_markers)
    _raw_no_button.on_click(_skip_raw_download)
    display(widgets.HBox([
        widgets.Label(f"{SAMPLE_NAME}'s raw H5AD is not cached. Download it now?"),
        _raw_yes_button, _raw_no_button,
    ]))
    display(_raw_download_output)

## 5 — Cell type co-occurrence (fraction correlation)

Pearson correlation of per-spot cell type fractions. High correlation = frequently co-localised.

In [ ]:
fractions = adata.obsm["cell_type_fractions"].rename(columns=strip_q05_abundance_prefix)
corr = fractions.corr()

fig, ax = plt.subplots(figsize=(max(8, len(cell_types_) * 0.6),
                                max(6, len(cell_types_) * 0.5)))
sns.heatmap(
    corr,
    cmap="coolwarm",
    center=0,
    vmin=-1, vmax=1,
    square=True,
    linewidths=0.5,
    ax=ax,
    cbar_kws={"shrink": 0.7},
)
# scanpy's sc.set_figure_params() leaves the matplotlib axes grid enabled,
# which otherwise draws extra grey lines across the heatmap's cells, on top
# of (and separate from) the intentional white linewidths=0.5 cell borders.
ax.grid(False)
ax.set_title("Cell type fraction correlation across spots")

for tick_label in ax.get_xticklabels():
    tick_label.set_color(uber_palette[uber_type_of(tick_label.get_text())])
for tick_label in ax.get_yticklabels():
    tick_label.set_color(uber_palette[uber_type_of(tick_label.get_text())])

plt.tight_layout()
plt.show()